In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast

df = pd.read_csv('rj_listings.csv') #RJ como cidade escolhida

In [ ]:
print(df.head())


In [ ]:
df.columns.to_list()

In [ ]:
df.info()

In [ ]:
#Colunas que possuem nulos
nulos_por_coluna = df.isnull().sum()
print(nulos_por_coluna[nulos_por_coluna > 0])

In [ ]:
df['price'].dtype #Checar o tipo da coluna

In [ ]:
#remove $ para transformar posteriormente em float
df['price'] = df['price'].str.replace('$', '', regex=False) 
#remove , para transformar posteriormente em float
df['price'] = df['price'].str.replace(',', '', regex=False)

In [ ]:
#Transforma em Float
df['price'] = df['price'].astype(float)

In [ ]:
df['price'].describe()

In [ ]:
df['amenities'].describe()

In [ ]:
#Descobre as top 5 amenidades para criar categorias binarias
pd.Series([item for sublist in df['amenities'].dropna().apply(ast.literal_eval) for item in sublist]).value_counts().head(5)

As cinco amenidades escolhidas foram Wifi, Kitchen, TV, Hot water e Air conditioning, pois apresentaram as maiores frequências na base. Dessa forma, a análise concentra-se nas comodidades mais comuns entre os anúncios da cidade.

In [ ]:
top_amenidades = [
    'Wifi',
    'Kitchen',
    'TV',
    'Hot water',
    'Air conditioning'
] #Cria uma coluna para cada uma das 5 maiores amenidades

for amenidade in top_amenidades:
    df[amenidade] = df['amenities'].apply(
        lambda x: 1 if amenidade in x else 0
    )

In [ ]:
df[top_amenidades].head()

Pergunta 1 a ser respondida: Quais amenidades estão mais associadas ao preço das acomodações?

In [ ]:
#para cada uma das 5 amenidades, o preço mediano dos anúncios que possuem a 
#amenidade com o preço mediano dos anúncios que não possuem.
for amenidade in top_amenidades:
    results = df.groupby(amenidade)['price'].agg(
        ['count', 'mean', 'median']
    )

    print(f'\n {amenidade} ')
    print(results)

Quanto a mediana do preco muda comparando acomodacoes que tem determinada amenidade com as que nao tem.

In [ ]:
#Quanto o preço mediano varia, em termos percentuais, entre acomodações que 
#possuem determinada amenidade e acomodações que não possuem?
percentuais = {}

for amenidade in top_amenidades:
    results = df.groupby(amenidade)['price'].agg(
        ['count', 'mean', 'median']
    )

    mediana_sem = results.loc[0, 'median']
    mediana_com = results.loc[1, 'median']

    percentuais[amenidade] = (
        (mediana_com - mediana_sem) / mediana_sem
    ) * 100

percentuais = pd.Series(percentuais).sort_values(ascending=False)

percentuais

In [ ]:
percentuais.plot(kind='bar', color='purple')

plt.xlabel('Amenidade')
plt.ylabel('Diferença percentual da mediana (%)')
plt.title('Diferença percentual do preço mediano por amenidade')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Análise aprofundada: A associação entre as amenidades e o preço das acomodações permanece quando controlamos por características da acomodação?

In [ ]:
#Comparação entre acomodações com e sem TV
df['has_tv'] = df['amenities'].str.lower().str.contains('tv', na=False)
df['has_tv'] = df['has_tv'].map({True: 'Com TV', False: 'Sem TV'})
tabela_preco= pd.pivot_table(
    df,
    values='price', 
    index=['bedrooms'],
    columns=['has_tv'], 
    aggfunc='median' 
).round(2)
tabela_preco.index = tabela_preco.index.astype(int)
tabela_preco['Diferença (R$)'] = tabela_preco['Com TV'] - tabela_preco['Sem TV']
tabela_preco = tabela_preco.fillna('-')

tabela_volume = pd.pivot_table(
    df,
    values='price', 
    index=['bedrooms'],
    columns=['has_tv'],
    aggfunc='count' 
).fillna(0)
tabela_volume.index = tabela_volume.index.astype(int)
display(tabela_preco.head(7)) 
display(tabela_volume.head(7))

Pode se notar que, excluindo os estudios(bedrooms=0), há uma disparidade de preço meio inconsistente comparando exemplos com e sem tv.   
Vale resaltar que a quantidade de locais sem tv é muito abaixo da de locais com tv.   


Locais com apenas um quarto são os que possuem mais dados, onde a mediana retorna que há uma diferença de R$99 reais entre quartos com e sem TV.   
De dois a quatro quartos, a presença da TV afeta menos o preço, considerando que quartos a mais já adicionam um bom peso ao preço.   

Os ambientes com 5+ quartos são questionaveis a analise, pode se extrair que a falta de uma TV indica que outras qualidades também não estão presentes, levando a grande diferença de preço, porem devido a quantidade de dados, não pode se extrair muito mais do que isto.

In [ ]:
tabela_grafico = pd.pivot_table(
    df[(df['bedrooms'] >= 1) & (df['bedrooms'] <= 4)],
    values='price', 
    index=['bedrooms'],
    columns=['has_tv'], 
    aggfunc='median' 
)
tabela_grafico.index = tabela_grafico.index.astype(int)
ax = tabela_grafico.plot(kind='bar', width=0.5,color=["#28a3c9","#c94d28"])
ax.set_facecolor('#ededed') 
ax.figure.patch.set_facecolor('#ededed')
plt.title('Preço mediano por numero de quartos', fontsize=14)
plt.xlabel('Numero de quartos', fontsize=14)
plt.ylabel('Preço mediano (R$)', fontsize=14)
plt.legend(title='Local')
plt.xticks(rotation=0)
plt.show()

Conseguimos evidenciar melhor a discrepancia de preço e dar luz ao que foi abordado na tabela anterior, elminando estudios e locais com mais de 4 quartos, é possivel ver uma crescente estável que mantém os padrões de locais sem tv serem mais baratos.

In [ ]:
def analisar_combo_amenidades(df, lista_amenidades, nome_pacote):
    df_temp = df.copy()
    condicao_combo = pd.Series(True, index=df_temp.index)
    #classificando caso tem a amenidade ou não
    for amenidade in lista_amenidades:
        tem = df_temp['amenities'].str.lower().str.contains(amenidade.lower(), na=False)
        condicao_combo = condicao_combo & tem
        
    #coluna combo
    nome_coluna = 'Amenidades'
    df_temp[nome_coluna] = condicao_combo.map({
        True: f'contem as {nome_pacote}', 
        False: f'Não contem as {nome_pacote}'
    })
    
    #Filtro
    df_grafico = df_temp[
        (df_temp['bedrooms'] >= 1) &     
        (df_temp['bathrooms'] >= 1) &              
        (df_temp['room_type'] != 'Hotel room') &   
        (df_temp['room_type'] != 'Shared room') &    
        (df_temp['bedrooms'] <= 4) & 
        (df_temp['bathrooms'] <= 3)
    ]
    
    g = sns.catplot(
        data=df_grafico,
        x='bedrooms',           
        y='price',              
        hue=nome_coluna,
        col='room_type',        
        row='bathrooms',        
        kind='bar',             
        estimator='median',     
        errorbar=None,          
        height=4,               
        aspect=1.2,
        sharey='row',
        palette=["#c94d28","#28a3c9"],
        width=0.7          
    )

    g.set_axis_labels("Número de Quartos", "Preço Mediano (R$)")
    g.set_titles("Tipo: {col_name} | Banheiros: {row_name}")
    g.fig.patch.set_facecolor('#ededed')

    for ax in g.axes.flatten():
        ax.tick_params(labelbottom=True)
        ax.set_facecolor('#ededed')
        
    g.fig.suptitle(f'Comparação de custo, imóveis que incluem as amenidades "tv, cozinha e ar condicionado"', y=1)
    sns.move_legend(g, "center right", bbox_to_anchor=(1.25, 0.5))
    plt.tight_layout(rect=[0, 0, 0.9, 1])
    plt.tight_layout()
    plt.show()

minhas_amenidades = ['tv', 'kitchen', 'air conditioning']
analisar_combo_amenidades(df_analise, minhas_amenidades, 'todas amenidades')

Já este gráfico divide melhor onde e como o preço está alocado, dependendo da quartidade de banheiros e quartos. Outra diferença é que a linha azul é um conjunto das 3 amenidades que influenciam mais o preço, neste caso tv, cozinha e ar condicionado.   

Outra aviso sobre o gráfico é que, para ter uma visualização melhor, o eixo Y muda por apenas por linha, então a linha 1 utiliza os mesmos numeros para a escala do y.

É possível observar que, embora tenha alguns fora da curva, a maioría dos casos demonstra que o preço das amenidades influencia os locais com menor numero de quartos. Quanto mais alto o número de quartos, mais o preço tende a depender do local em si, principalmente os locais com 4 quartos.

Embora o gráfco de preço mediano por quartos mostre que locais com tv são mais caros de 1 a 4 quartos, adicionar o numero de banheiros e fazer a distinção entre "private rooms" e imoveis completos muda a situação.

A escala de preço por amenidades se mantém para locais com poucos quartos e banheiros, porém quanto maior o numero de quarto/banheiro, mais comum ter uma inconsistencia não explicada por amenidades.